# Micronic

PyLabRobot includes `v1b1` Micronic integrations built on the generic
`rack_reading` capability.

`MicronicCodeReader` controls the local hardware directly. The caller picks a
`Scanner` to acquire the rack image (`TwainScanner` on Windows, `SaneScanner` on
Linux, or `CommandScanner` for any other acquisition stack); the driver reads
the side rack barcode through the serial reader, decodes tube DataMatrix codes
locally, and returns a standard `RackScanResult`. It does not call Micronic
Code Reader or IO Monitor, and PyLabRobot does not package any scanner helper
executable.

## Supported operations

Rack reading (large scanner that decodes 96 tubes plus the side rack barcode):

- `rack_reading.scan_rack(rack)` to trigger image acquisition, decode all 96 tube
  positions, read the side rack barcode, and return a `RackScanResult`. The driver
  only supports 8x12 racks; passing a different shape raises `MicronicError`.
- `rack_reading.scan_rack_id()` for a rack-barcode-only read on the side reader

## Hardware example

The operator is responsible for installing any OS-level scanner bridge
(`twain_scan`, `scanimage`, or a custom command), the PLR serial extra
(`pylabrobot[serial]`), and the local Python decode dependencies in the runtime
environment.

In [ ]:
from pylabrobot.micronic import MicronicCodeReader, TwainScanner

reader = MicronicCodeReader(
  scanner=TwainScanner(
    twain_scanner_path=r"C:\Tools\twain_scan.exe",
    twain_source="AVA6PlusG",
  ),
  serial_port="COM4",
  image_dir=r"C:\ProgramData\PyLabRobot\micronic-images",
  keep_images=True,
)
await reader.setup()

try:
  rack_result = await reader.rack_reading.scan_rack(
    rack=my_rack, timeout=90.0, poll_interval=1.0
  )
  print(rack_result.rack_id)
  print(len([entry for entry in rack_result.entries if entry.tube_id]))

  rack_id = await reader.rack_reading.scan_rack_id(timeout=5.0, poll_interval=0.5)
  print(rack_id)
finally:
  await reader.stop()

On Ubuntu/Linux, use `SaneScanner` if the scanner is exposed by a SANE backend:

In [ ]:
from pylabrobot.micronic import MicronicCodeReader, SaneScanner

reader = MicronicCodeReader(
  scanner=SaneScanner(sane_device="avision:libusb:001:004"),
  serial_port="/dev/ttyUSB0",
)

For any other acquisition stack, subclass `Scanner` and implement `acquire` to
write a rack image to the given path.

## Notes

- `scan_rack` reads every tube barcode and finishes by reading the rack ID, so
  it typically takes tens of seconds. `scan_rack_id` only reads the rack
  barcode and completes in a few seconds.
- TWAIN is a Windows scanner-driver API. PyLabRobot does not ship a TWAIN
  bridge binary and does not install one for you; pass `twain_scanner_path` to
  `TwainScanner`, set `MICRONIC_TWAIN_SCANNER_PATH`, or put a local helper
  named `twain_scan` / `twain_scan.exe` on `PATH`.
- Ubuntu/Linux scanner control uses SANE `scanimage` (via `SaneScanner`).
  PyLabRobot does not install SANE or vendor scanner drivers. Rack-ID reads use
  `pylabrobot.io.Serial`, which is installed through the `pylabrobot[serial]`
  extra.
- Image decoding imports `pillow`, `opencv-python-headless`, `numpy`, and
  `zxing-cpp` at runtime. Install them in the environment that runs PyLabRobot.